[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C01_LLM_Internals_Course/02_attention_transformer/02_attention_transformer.ipynb)

# 02 · 手写 Attention 与 Transformer Block

<span style="background:#234;color:#9fd;padding:2px 8px;border-radius:4px">CPU</span> 纯 PyTorch，无需 GPU、无需下载任何模型。

本 notebook 是模块 02 讲解（`02_讲解.html`）的动手部分。我们从一行公式出发，逐步搭出一个完整的 GPT 骨架，
并且**每一步都与 PyTorch 官方实现数值对拍**——写错任何一个维度、漏掉 $\sqrt{d_k}$ 缩放，assert 都会立刻报警。

**路线图**：
1. 手写 `scaled_dot_product_attention`（含 causal mask）→ 与 `F.scaled_dot_product_attention` 对拍
2. 手写 `MultiHeadAttention` → 与 `nn.MultiheadAttention`（拷贝同一份权重）对拍
3. `FeedForward` + Pre-LN `Block` [Xiong 2020] → 组装 `MiniGPT`（权重 tying）
4. 逐层形状追踪、attention 热图、参数量与手算公式对照
5. ✏️ 3 道练习 + 📖 参考答案

参考：[Vaswani 2017] *Attention Is All You Need*；[Xiong 2020] *On Layer Normalization in the Transformer Architecture*；[Karpathy 2022] *nanoGPT*。

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
print('torch:', torch.__version__)

## 1. 手写 scaled dot-product attention

核心公式（推导见讲解 §2）：

$$\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\Big(\frac{QK^\top}{\sqrt{d_k}}+M\Big)V,\qquad M_{ts}=\begin{cases}0 & s\le t\\ -\infty & s>t\end{cases}$$

三个实现要点：
- **缩放用每个头的维度** $d_k$（张量最后一维），不是模型宽度；
- **softmax 沿最后一维**（key 维）：每个 query 的权重构成一个概率分布；
- **causal mask 在 softmax 之前**以 $-\infty$ 加法实现：$e^{-\infty}=0$，被屏蔽位置权重精确为零且剩余权重自动归一。

写完立刻与 PyTorch 官方 `F.scaled_dot_product_attention` 数值对拍——这是本课贯穿始终的习惯：**自己的实现必须能通过与权威实现的 allclose**。

In [ ]:
def scaled_dot_product_attention(q, k, v, causal=False):
    # q, k, v: [B, H, T, d_k]   返回: [B, H, T, d_k]
    d_k = q.size(-1)
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)   # [B, H, T, T]
    if causal:
        T = q.size(-2)
        mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=q.device))
        scores = scores.masked_fill(~mask, float('-inf'))  # 上三角 -> -inf
    weights = F.softmax(scores, dim=-1)                  # 沿 key 维归一
    return weights @ v

# ---- 与 PyTorch 官方实现对拍 ----
B, H, T, d_k = 2, 4, 16, 32
q, k, v = (torch.randn(B, H, T, d_k) for _ in range(3))

for causal in (False, True):
    ours = scaled_dot_product_attention(q, k, v, causal=causal)
    ref  = F.scaled_dot_product_attention(q, k, v, is_causal=causal)
    assert ours.shape == ref.shape == (B, H, T, d_k)
    assert torch.allclose(ours, ref, atol=1e-6), f'causal={causal} 数值不一致'
    print(f'causal={causal}: max|diff| = {(ours - ref).abs().max().item():.2e}  ✓ allclose')

print('✅ 手写 SDPA 与 F.scaled_dot_product_attention 对拍通过')

## 2. 多头注意力 MultiHeadAttention

多头 = 把 $C$ 维表征切成 $h$ 个 $d_h=C/h$ 维子空间，各自独立做一套 attention，拼回后过输出投影 $W_O$（讲解 §4）。

实现技巧（nanoGPT 风格）：
- $W_Q, W_K, W_V$ 融合成**一个** `Linear(C, 3C)`，一次矩阵乘后 `split`；
- 切头只是 `view` + `transpose`：`[B,T,C] → [B,T,h,d_h] → [B,h,T,d_h]`，**不增加计算量**；
- 参数量 $= 4C^2+4C$，**与头数无关**。

对拍对象换成 `nn.MultiheadAttention(batch_first=True)`：把**同一份权重**拷贝给官方实现，输出必须 allclose。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_head):
        super().__init__()
        assert n_embd % n_head == 0, 'n_embd 必须能被 n_head 整除'
        self.n_head = n_head
        self.head_dim = n_embd // n_head
        self.qkv  = nn.Linear(n_embd, 3 * n_embd)   # W_Q,W_K,W_V 融合
        self.proj = nn.Linear(n_embd, n_embd)       # W_O
        self.attn_weights = None                    # 最近一次前向的权重 [B,h,T,T]，供可视化

    def forward(self, x, causal=True):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)                            # 3 x [B,T,C]
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)     # [B,h,T,d_h]
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)      # [B,h,T,T]
        if causal:
            mask = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
            scores = scores.masked_fill(~mask, float('-inf'))
        w = F.softmax(scores, dim=-1)
        self.attn_weights = w.detach()
        out = w @ v                                                      # [B,h,T,d_h]
        out = out.transpose(1, 2).contiguous().view(B, T, C)             # 合头 -> [B,T,C]
        return self.proj(out)

# ---- 与 nn.MultiheadAttention 对拍（拷贝同一份权重）----
C, h, B, T = 64, 4, 2, 12
mha = MultiHeadAttention(C, h)
ref = nn.MultiheadAttention(C, h, batch_first=True)
with torch.no_grad():
    ref.in_proj_weight.copy_(mha.qkv.weight)   # 官方把 W_Q,W_K,W_V 也按行堆叠
    ref.in_proj_bias.copy_(mha.qkv.bias)
    ref.out_proj.weight.copy_(mha.proj.weight)
    ref.out_proj.bias.copy_(mha.proj.bias)

x = torch.randn(B, T, C)
# 非因果
out_ref, _ = ref(x, x, x, need_weights=False)
assert torch.allclose(mha(x, causal=False), out_ref, atol=1e-5)
# 因果：官方用布尔上三角 attn_mask（True = 屏蔽）
causal_attn_mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
out_ref, _ = ref(x, x, x, attn_mask=causal_attn_mask, need_weights=False)
assert torch.allclose(mha(x, causal=True), out_ref, atol=1e-5)

n_params = sum(p.numel() for p in mha.parameters())
assert n_params == 4 * C * C + 4 * C, 'MHA 参数量应为 4C^2+4C'
print(f'MHA 参数量 = {n_params} = 4*{C}^2 + 4*{C}  （与头数 h={h} 无关）')
print('✅ MultiHeadAttention 与 nn.MultiheadAttention 对拍通过（causal / 非 causal）')

## 3. FeedForward、Pre-LN Block 与 MiniGPT 组装

**FeedForward**：$C \to 4C \to \mathrm{GELU} \to C$，逐位置独立作用，参数量 $8C^2+5C$——块内参数大头（讲解 §5）。

**Block 用 Pre-LN**（GPT-2 / nanoGPT 及之后的标准做法）：

$$x \leftarrow x + \mathrm{MHA}(\mathrm{LN}_1(x)),\qquad x \leftarrow x + \mathrm{FFN}(\mathrm{LN}_2(x))$$

残差主干是一条不被打断的恒等路径，梯度尺度健康，无需 warmup 也能稳定训练 [Xiong 2020]；
代价是必须在所有 block 之后补一个最终 `ln_f`。

**MiniGPT** = token embedding + position embedding + $L$ 个 Block + `ln_f` + `lm_head`。
其中 `lm_head` 与 token embedding **共享同一份权重**（weight tying，GPT-2/nanoGPT 默认）：
省一份 $V\times C$ 参数，并强制输入词向量空间与输出预测空间一致。

`forward(verbose=True)` 会逐层打印张量形状——对照讲解 §7 的形状追踪表。

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),   # 升维 4x
            nn.GELU(),
            nn.Linear(4 * n_embd, n_embd),   # 降回
        )
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    # Pre-LN: LN 在子层之前，残差主干恒等直通
    def __init__(self, n_embd, n_head):
        super().__init__()
        self.ln1  = nn.LayerNorm(n_embd)
        self.attn = MultiHeadAttention(n_embd, n_head)
        self.ln2  = nn.LayerNorm(n_embd)
        self.ffn  = FeedForward(n_embd)
    def forward(self, x):
        x = x + self.attn(self.ln1(x), causal=True)   # 残差: 读总线 -> 写增量
        x = x + self.ffn(self.ln2(x))
        return x

blk = Block(64, 4)
x = torch.randn(2, 12, 64)
assert blk(x).shape == x.shape, 'Block 必须保持 [B,T,C] 不变（残差流总线宽度恒定）'
ffn_params = sum(p.numel() for p in blk.ffn.parameters())
assert ffn_params == 8 * 64 * 64 + 5 * 64
print('FFN 参数量 =', ffn_params, '= 8C^2+5C  （约为 MHA 的 2 倍 → 参数大头）')
print('✅ Block 前向形状检查通过')

In [ ]:
class MiniGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer, n_head, n_embd):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)    # learned 位置编码（讲解 §8）
        self.blocks  = nn.ModuleList([Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f    = nn.LayerNorm(n_embd)                # Pre-LN 架构必需的最终 LN
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight          # ---- weight tying ----

    def forward(self, idx, verbose=False):
        B, T = idx.shape
        assert T <= self.block_size
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)          # [B,T,C]
        if verbose:
            print(f'{"input ids":<28} {list(idx.shape)}')
            print(f'{"tok_emb + pos_emb":<28} {list(x.shape)}')
        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if verbose:
                aw = blk.attn.attn_weights
                print(f'{f"block {i} 输出":<28} {list(x.shape)}   attn weights {list(aw.shape)}')
        x = self.ln_f(x)
        logits = self.lm_head(x)                           # [B,T,V]
        if verbose:
            print(f'{"logits":<28} {list(logits.shape)}')
        return logits

cfg = dict(vocab_size=256, block_size=64, n_layer=4, n_head=4, n_embd=128)
model = MiniGPT(**cfg)

# weight tying 验证：两个名字指向同一块内存，参数只算一份
assert model.lm_head.weight.data_ptr() == model.tok_emb.weight.data_ptr()
print('weight tying: lm_head.weight 与 tok_emb.weight 共享内存 ✓\n')

# 逐层形状追踪（对照讲解 §7 的表）
idx = torch.randint(0, cfg['vocab_size'], (2, 16))
logits = model(idx, verbose=True)
assert logits.shape == (2, 16, cfg['vocab_size'])
print('\n✅ MiniGPT 组装完成，形状全程 [B,T,C]，仅 attention 分数为 [B,h,T,T]')

## 4. Attention 权重热图

每个头的权重矩阵 `[T,T]` 第 $t$ 行是位置 $t$ 对所有位置的注意力分布。两点观察：

- **上三角严格为 0**——causal mask 在起作用，任何一个非零都意味着未来信息泄漏；
- 现在权重是**随机初始化**下的样子：行内分布接近均匀、无结构。**模块 03 训练之后**，
  这里会浮现清晰的模式（关注前一个 token 的位置头、检索重复前缀的 induction head 等）——届时回来对比这张图。

In [ ]:
idx = torch.randint(0, cfg['vocab_size'], (1, 32))
_ = model(idx)                                  # 前向一次，缓存各层 attn_weights
w = model.blocks[0].attn.attn_weights[0]        # block 0 的权重 [h, T, T]

fig, axes = plt.subplots(1, cfg['n_head'], figsize=(3.2 * cfg['n_head'], 3.2))
for i, ax in enumerate(axes):
    im = ax.imshow(w[i], cmap='viridis', vmin=0)
    ax.set_title(f'block 0 / head {i}')
    ax.set_xlabel('key 位置 s')
    if i == 0:
        ax.set_ylabel('query 位置 t')
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

# 数值验证：上三角必须精确为 0（causal），每行和为 1（softmax）
assert torch.all(w.triu(1) == 0), 'causal mask 泄漏！'
assert torch.allclose(w.sum(-1), torch.ones_like(w.sum(-1)), atol=1e-6)
print('✅ 上三角全 0（无未来泄漏）、每行归一')

## 5. 参数量：逐组件统计 vs 手算公式

讲解 §5 的公式（含 bias，$C$=n_embd，$L$=n_layer，$V$=vocab，$T_{max}$=block_size）：

| 组件 | 公式 |
|---|---|
| MHA / 层 | $4C^2+4C$ |
| FFN / 层 | $8C^2+5C$ |
| 2×LN / 层 | $4C$ |
| tok_emb（与 lm_head tying，只算一份） | $VC$ |
| pos_emb | $T_{max}C$ |
| ln_f | $2C$ |

下面按参数名分桶统计真实模型，并 **assert 与公式逐项一致**。

In [ ]:
def param_breakdown(model):
    buckets = {'embedding': 0, 'attention': 0, 'ffn': 0, 'layernorm': 0}
    for name, p in model.named_parameters():   # named_parameters 自动对 tied 权重去重
        if 'tok_emb' in name or 'pos_emb' in name or 'lm_head' in name:
            buckets['embedding'] += p.numel()
        elif '.attn.' in name:
            buckets['attention'] += p.numel()
        elif '.ffn.' in name:
            buckets['ffn'] += p.numel()
        else:
            buckets['layernorm'] += p.numel()
    return buckets

C, L = cfg['n_embd'], cfg['n_layer']
V, Tm = cfg['vocab_size'], cfg['block_size']
bk = param_breakdown(model)
total = sum(p.numel() for p in model.parameters())

print(f'{"组件":<12}{"实测":>10}{"公式":>14}')
rows = [('embedding', V*C + Tm*C,        'VC + Tmax*C（tying 只算一份）'),
        ('attention', L*(4*C*C + 4*C),   'L(4C^2+4C)'),
        ('ffn',       L*(8*C*C + 5*C),   'L(8C^2+5C)'),
        ('layernorm', L*4*C + 2*C,       'L*4C + 2C')]
for name, formula_val, formula in rows:
    print(f'{name:<12}{bk[name]:>10}{formula_val:>14}   {formula}')
    assert bk[name] == formula_val, f'{name} 与公式不符'

print(f'{"总计":<12}{total:>10}{sum(r[1] for r in rows):>14}')
assert total == sum(r[1] for r in rows)
print(f'\nFFN 占 Transformer 主体（attn+ffn+ln）的比例: '
      f'{bk["ffn"] / (bk["attention"] + bk["ffn"] + bk["layernorm"]):.1%}')
print('✅ 逐组件参数量与手算公式全部一致')

## ✏️ 练习 1：实现 `causal_mask(T)`

实现一个函数，返回形状 `[T, T]` 的**下三角布尔矩阵**：`mask[t, s] == True` 表示位置 $t$ 可以看位置 $s$（即 $s\le t$）。

提示：
- `torch.tril` / `torch.ones` / `dtype=torch.bool`，一两行即可；
- 也可以用 `torch.arange` 广播比较（`行索引 >= 列索引`）自己造一个；
- 这正是 SDPA 里 `masked_fill(~mask, -inf)` 用的那个 mask。

In [ ]:
def causal_mask(T):
    # 返回 [T, T] 的 bool 下三角矩阵：mask[t, s] = (s <= t)
    # TODO: 用 torch.tril 或 arange 广播实现
    raise NotImplementedError

In [ ]:
m = causal_mask(5)
assert m.shape == (5, 5)
assert m.dtype == torch.bool
assert m[0, 0].item() is True and m[0, 1].item() is False
assert m[4].all(), '最后一行应全可见'
assert int(m.sum()) == 5 * 6 // 2, '下三角应有 T(T+1)/2 个 True'
assert torch.equal(m, torch.tril(torch.ones(5, 5, dtype=torch.bool)))
# 边界：T=1
m1 = causal_mask(1)
assert m1.shape == (1, 1) and m1[0, 0].item() is True
print('✅ 练习 1 通过')

## ✏️ 练习 2：单头 attention 的逐步版本

不用任何现成 attention 函数，对**单头**输入 `q, k, v`（形状均为 `[T, d]`）显式写出三步，
返回 `(scores, weights, out)` 三元组：

1. `scores = q @ k.T / sqrt(d)`，再把上三角（未来位置）填成 `-inf`（可复用练习 1 的 `causal_mask`）；
2. `weights = softmax(scores, dim=-1)`；
3. `out = weights @ v`。

自测会检查：每行权重和为 1、上三角权重为 0、`out` 与第 1 节的向量化版 `scaled_dot_product_attention` allclose。

In [ ]:
def attention_stepwise(q, k, v):
    # q, k, v: [T, d]（单头、单样本）。返回 (scores, weights, out)
    T, d = q.shape
    # TODO step 1: scores = q @ k.T / sqrt(d)，上三角填 -inf
    # TODO step 2: weights = softmax(scores, dim=-1)
    # TODO step 3: out = weights @ v
    raise NotImplementedError

In [ ]:
torch.manual_seed(0)
T, d = 6, 8
q, k, v = torch.randn(T, d), torch.randn(T, d), torch.randn(T, d)
scores, weights, out = attention_stepwise(q, k, v)

assert scores.shape == weights.shape == (T, T) and out.shape == (T, d)
upper = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
assert torch.isinf(scores[upper]).all() and (scores[upper] < 0).all(), '上三角分数应为 -inf'
assert torch.allclose(weights.sum(-1), torch.ones(T), atol=1e-6), '每行权重应和为 1'
assert torch.all(weights.triu(1) == 0), '未来位置权重应为 0'
# 与第 1 节的向量化实现对拍（加上 batch/head 维）
ref = scaled_dot_product_attention(q[None, None], k[None, None], v[None, None], causal=True)[0, 0]
assert torch.allclose(out, ref, atol=1e-6)
print('✅ 练习 2 通过')

## ✏️ 练习 3：闭式参数量公式 `count_params(cfg)`

只用 `cfg` 字典（不构造模型）算出 MiniGPT 的总参数量，必须与 `sum(p.numel())` 完全相等。

提示（第 5 节的表已给出全部零件，注意三个坑）：
- `lm_head` 与 `tok_emb` **tying**，只算一份 $VC$；
- 每层 $= 4C^2+4C$（attn）$+\,8C^2+5C$（ffn）$+\,4C$（两个 LN）；
- 别忘了 `pos_emb` 的 $T_{max}C$ 和最后的 `ln_f`（$2C$）。

In [ ]:
def count_params(cfg):
    # cfg: dict(vocab_size, block_size, n_layer, n_head, n_embd) -> int 总参数量
    V, Tm = cfg['vocab_size'], cfg['block_size']
    L, C = cfg['n_layer'], cfg['n_embd']
    # TODO: embedding 部分 + L * (每层 attn/ffn/ln) + ln_f
    raise NotImplementedError

In [ ]:
assert count_params(cfg) == sum(p.numel() for p in model.parameters())
# 换一组超参再验，防止硬编码
cfg2 = dict(vocab_size=100, block_size=32, n_layer=2, n_head=2, n_embd=64)
model2 = MiniGPT(**cfg2)
assert count_params(cfg2) == sum(p.numel() for p in model2.parameters())
print(f'count_params(cfg)  = {count_params(cfg):,}')
print(f'count_params(cfg2) = {count_params(cfg2):,}')
print('✅ 练习 3 通过')

## 📖 参考答案

先自己做，再对照。

In [ ]:
# 参考答案：练习 1（先自己做，再对照）
def causal_mask(T):
    return torch.tril(torch.ones(T, T, dtype=torch.bool))
    # 等价写法：torch.arange(T)[:, None] >= torch.arange(T)[None, :]

In [ ]:
# 参考答案：练习 2（先自己做，再对照）
def attention_stepwise(q, k, v):
    T, d = q.shape
    scores = q @ k.T / math.sqrt(d)                          # [T,T] 相似度
    scores = scores.masked_fill(~causal_mask(T), float('-inf'))
    weights = F.softmax(scores, dim=-1)                      # 每行一个概率分布
    out = weights @ v                                        # value 的凸组合
    return scores, weights, out

In [ ]:
# 参考答案：练习 3（先自己做，再对照）
def count_params(cfg):
    V, Tm = cfg['vocab_size'], cfg['block_size']
    L, C = cfg['n_layer'], cfg['n_embd']
    emb       = V * C + Tm * C                 # tok_emb(tying 只算一份) + pos_emb
    per_layer = (4*C*C + 4*C) + (8*C*C + 5*C) + 4*C   # attn + ffn + 2xLN
    return emb + L * per_layer + 2 * C         # + ln_f

## 小结

本 notebook 你完成了：

- 手写 `scaled_dot_product_attention`（$\sqrt{d_k}$ 缩放 + $-\infty$ causal mask），与 `F.scaled_dot_product_attention` 数值对拍；
- 手写 `MultiHeadAttention`（fused QKV、view/transpose 切头），与拷贝同权重的 `nn.MultiheadAttention` 对拍；
- 用 Pre-LN [Xiong 2020] 组装 `Block` 与 `MiniGPT`（learned 位置编码 + weight tying），逐层验证形状；
- 把参数量拆到组件级并与闭式公式 assert 一致——确认 **FFN 占主体参数约 2/3**，整模型 $\approx 12LC^2+(V+T_{max})C$。

这个 `MiniGPT` 目前还是随机权重——attention 热图毫无结构、logits 接近均匀。
**模块 03（`../03_train_minigpt/03_讲解.html`）**将在真实文本上训练它：交叉熵目标、AdamW、学习率调度，
训练后回来重画 §4 的热图，你会看到位置头与 induction head 浮现出来。

---
## 🎯 真实数据胶囊题：真实句子上的因果自注意力

把一句真实英文按字符 tokenize，用随机但固定的 embedding 算**因果**缩放点积注意力。验证注意力的两个铁律：每行权重和为 1、因果 mask 让位置 i 看不到未来。

> 本题为本模块新增的**真实数据**练习：自包含，直接用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, urllib.request, numpy as np
CACHE=os.path.expanduser("~/.llm_internals_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def shakespeare():
    return open(_fetch("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt","shake.txt")).read()

sent = "to be or not to be"   # 真实台词
chars = sorted(set(sent)); stoi={c:i for i,c in enumerate(chars)}
ids = np.array([stoi[c] for c in sent])
rng=np.random.default_rng(0); d=16
E = rng.normal(size=(len(chars), d))     # 字符 embedding
X = E[ids]                                # (T,d) 真实序列的表示
print("序列长度 T=", len(ids))

**练习**：实现 `causal_attention(X)`：返回 `(输出, 注意力权重)`。用 scaled dot-product，加因果 mask（上三角设 -inf），softmax 按行。

In [ ]:
def causal_attention(X):
    # TODO: S=X@X.T/sqrt(d); 上三角(不含对角)设 -inf; 行 softmax; 输出=W@X
    raise NotImplementedError


In [ ]:
# 自测
out, W = causal_attention(X)
assert out.shape == X.shape
assert np.allclose(W.sum(1), 1.0), "每行权重和为1"
assert np.allclose(np.triu(W,1), 0), "因果：不能看未来(上三角为0)"
assert W[0,0]==1.0, "第一个 token 只能看自己"
print("因果自注意力两条铁律验证通过 ✓")


### 📖 参考答案

In [ ]:
def causal_attention(X):
    T,d=X.shape; S=X@X.T/np.sqrt(d)
    mask=np.triu(np.ones((T,T)),1).astype(bool); S=np.where(mask,-np.inf,S)
    S=S-S.max(1,keepdims=True); W=np.exp(S); W/=W.sum(1,keepdims=True)
    return W@X, W
print("✓ 因果 mask + 行归一化 = 自回归注意力的核心")